In [1]:
import numpy as np
import pandas as pd
import os
import textwrap

In [2]:
def load_data(data_dir, x_filename='x_train.csv', y_filename='y_train.csv'):
    x_train_df = pd.read_csv(os.path.join(data_dir, x_filename))
    y_train_df = pd.read_csv(os.path.join(data_dir, y_filename))

    N, n_cols = x_train_df.shape
    print("Shape of x_train_df: (%d, %d)" % (N, n_cols))
    print("Shape of y_train_df: %s" % str(y_train_df.shape))

    # Print out 8 random entries
    tr_text_list = x_train_df['text'].values.tolist()
    prng = np.random.RandomState(101)
    rows = prng.permutation(np.arange(y_train_df.shape[0]))
    for row_id in rows[:8]:
        text = tr_text_list[row_id]
        print("row %5d | %s BY %s | y = %s" % (
            row_id,
            y_train_df['title'].values[row_id],
            y_train_df['author'].values[row_id],
            y_train_df['Coarse Label'].values[row_id],
            ))
        # Pretty print text via textwrap library
        line_list = textwrap.wrap(tr_text_list[row_id],
            width=70,
            initial_indent='  ',
            subsequent_indent='  ')
        print('\n'.join(line_list))
        print("")
    return x_train_df, y_train_df

In [3]:
x_train_df, y_train_df = load_data(data_dir = "data", x_filename = "x_train.csv", y_filename = "y_train.csv")

Shape of x_train_df: (5557, 32)
Shape of y_train_df: (5557, 5)
row  4746 | The Red and the Black: A Chronicle of 1830 BY Stendhal | y = Key Stage 4-5
  It was hermetically sealed; he was on the point of  fainting and
  remained for a long time leaning against the oak; then  with a
  staggering step he went to have another look at the gardener's
  ladder. The chain which he had once forced asunder--in, alas, such
  different  circumstances--had not yet been repaired. Carried away by
  a moment of  madness, Julien pressed it to his lips.

row  1250 | Cranford BY Elizabeth Cleghorn Gaskell | y = Key Stage 4-5
  Miss Pole, Miss Matty, and I, meanwhile attended to Miss Brown: and
  hard  work we found it to relieve her querulous and never-ending
  complaints. But if we were so weary and dispirited, what must Miss
  Jessie have been! Yet she came back almost calm as if she had gained
  a new strength. She  put off her mourning dress, and came in,
  looking pale and gentle,  thanking us each 

In [4]:
def tokenize_text(raw_text):
    ''' Transform a plain-text string into a list of tokens
    
    We assume that *whitespace* divides tokens.
    
    Args
    ----
    raw_text : string
    
    Returns
    -------
    list_of_tokens : list of strings
        Each element is one token in the provided text
    '''
    list_of_tokens = raw_text.split() # split method divides on whitespace by default
    for pp in range(len(list_of_tokens)):
        cur_token = list_of_tokens[pp]
        # Remove punctuation
        for punc in ['?', '!', '_', '.', ',', '"', '/']:
            cur_token = cur_token.replace(punc, "")
        # Turn to lower case
        clean_token = cur_token.lower()
        # Replace the cleaned token into the original list
        list_of_tokens[pp] = clean_token
    return list_of_tokens

In [5]:
training_text = x_train_df['text'].values.tolist()
tokenized_training_text = [tokenize_text(text) for text in training_text]
print("Example tokenized texts:")
for i, tokens in enumerate(tokenized_training_text[:5]):
    print(f"Text {i}: {tokens}")

Example tokenized texts:
Text 0: ['yes', 'what', 'sort', 'of', 'terms', 'was', 'he', 'on', 'with', 'the', 'guests—you', 'and', 'miss', 'norris', 'and', 'all', 'of', 'them', 'just', 'polite', 'and', 'rather', 'silent', 'you', 'know', 'keeping', 'himself', 'to', 'himself', 'we', "didn't", 'see', 'so', 'very', 'much', 'of', 'him', 'except', 'at', 'meals', 'we', 'were', 'here', 'to', 'enjoy', 'ourselves', 'and—well', 'he', "wasn't", 'he', "wasn't", 'there', 'when', 'the', 'ghost', 'walked', 'no', 'i', 'heard', 'mark', 'calling', 'for', 'him', 'when', 'he', 'went', 'back', 'to', 'the', 'house', 'i', 'expect', 'cayley', 'stroked', 'down', 'his', 'feathers', 'a', 'bit', 'and', 'told', 'him', 'that', 'girls', 'will', 'be', 'girls—hallo', 'here', 'we', 'are']
Text 1: ['perhaps', 'i', 'should', 'say', 'that', 'it', 'was', "mark's", 'private', 'plan', 'my', 'own', 'was', 'different', 'the', 'announcement', 'at', 'breakfast', 'went', 'well', 'after', 'the', 'golfing-party', 'had', 'gone', 'off', '

In [6]:
def build_vocabulary(tokenized_text_list, min, max):
  tok_count_dict = dict()

  for tokenized_text in tokenized_text_list:
    for token in tokenized_text:
      if token not in tok_count_dict:
        tok_count_dict[token] = 1 # Initialize count for new token
      else:
        tok_count_dict[token] += 1 # Increment count for existing token
  
  # Sort the tokens by their counts in descending order
  sorted_tokens = list(sorted(tok_count_dict, key=tok_count_dict.get, reverse=True))

  # # Print out the 10 most common tokens and their counts
  # for w in sorted_tokens[:10]:
  #   print("%5d %s" % (tok_count_dict[w], w))
  
  # # Print out the 10 least common tokens and their counts
  # for w in sorted_tokens[-10:]:
  #   print("%5d %s" % (tok_count_dict[w], w))

  conjunctions = [
      # Coordinating conjunctions
      "nor", "yet",

      # Subordinating conjunctions
      "after", "although", "as", "because", "before", "once", "since", "though", "unless",
      "until", "whenever", "whereas", "wherever", "whether", "while",

      # Correlative conjunctions
      "either", "neither"
  ]
  
  # Merge plural counts into singular (e.g., 'dogs' -> 'dog'), skip empty tokens
  for t in list(tok_count_dict.keys()):
    if not t:
      continue
    if len(t) > 1 and t.endswith('s') and t[:-1] in tok_count_dict:
      tok_count_dict[t[:-1]] += tok_count_dict[t]
      tok_count_dict[t] = 0

  vocab_list = [w for w in sorted_tokens if ((tok_count_dict[w] >= min and tok_count_dict[w] <= max and len(w) >= 6) or w in conjunctions)]
  
  return vocab_list, tok_count_dict

In [7]:
import sklearn

def make_logit_pipeline(C=1.0):
    pipeline = sklearn.pipeline.Pipeline(
        steps=[
         ('rescaler', sklearn.preprocessing.MinMaxScaler()),
         ('logit', sklearn.linear_model.LogisticRegression(solver="lbfgs", l1_ratio=0, C=C, max_iter=1000))
        ])
    
    # Return the constructed pipeline
    return pipeline

In [8]:
def find_best_vocab(tokenized_text, y_labels, x_df, 
                         min_freq_list=None, 
                         max_freq_list=None,
                         C_grid=None, 
                         n_splits=10, 
                         random_state=42,
                         verbose=False):
    """
    Tune vocabulary frequency bounds and C parameter for logistic regression.
    
    Args:
    ----
    tokenized_text : list of list of strings
        Tokenized training texts
    y_labels : pandas Series
        Target labels
    x_df : pandas DataFrame
        Feature DataFrame (original x_train)
    min_freq_range : tuple (start, stop, step)
        Range for minimum frequency threshold
    max_freq_range : tuple (start, stop, step)
        Range for maximum frequency threshold
    C_grid : array-like or None
        C values to test; if None, defaults to logspace(-4, 4, 17)
    n_splits : int
        Number of K-fold splits
    random_state : int
        Random seed
    verbose : bool
        Print intermediate results
    
    Returns:
    -------
    results : dict with keys:
        - 'best_vocab_list': vocabulary words
        - 'best_tok_count_dict': token counts
        - 'best_min': minimum frequency
        - 'best_max': maximum frequency
        - 'best_auc': best CV AUC for vocab tuning
        - 'best_C': best C value (if tuning C)
        - 'best_model_auc': best AUC with C tuning
        - 'all_vocab_results': list of (min, max, auc) tuples
    """
    
    if C_grid is None:
        C_grid = np.logspace(-4, 4, 17)
    
    vocab_results = []
    
    kf = sklearn.model_selection.KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    
    for min_freq in min_freq_list:
        for max_freq in max_freq_list:
            if min_freq > max_freq:
                continue
            
            vocab_list, tok_count_dict = build_vocabulary(tokenized_text, min_freq, max_freq)
            
            # Build feature matrix for this vocab
            vocab_dict = {w: i for i, w in enumerate(vocab_list)}
            vocab_cols = [f"v_{w}" for w in vocab_list]
            
            rows = []
            for text in x_df['text'].values:
                vec = np.zeros(len(vocab_list), dtype=int)
                for tok in tokenize_text(text):
                    if tok in vocab_dict:
                        vec[vocab_dict[tok]] += 1
                rows.append(vec)
            X_counts = np.vstack(rows)
            
            vocab_df = pd.DataFrame(X_counts, columns=vocab_cols, index=x_df.index)
            x_aug = pd.concat([x_df, vocab_df], axis=1)
            
            # CV evaluation with this vocab
            pl = make_logit_pipeline(C=1.0)
            aucs = []
            for train_index, val_index in kf.split(x_aug):
                x_train_fold = x_aug.iloc[train_index][vocab_cols]
                y_train_fold = y_labels.iloc[train_index]
                x_val_fold = x_aug.iloc[val_index][vocab_cols]
                y_val_fold = y_labels.iloc[val_index]
                
                pl.fit(x_train_fold, y_train_fold)
                y_pred_proba = pl.predict_proba(x_val_fold)[:, 1]
                auc = sklearn.metrics.roc_auc_score(y_val_fold, y_pred_proba)
                aucs.append(auc)
            
            mean_auc = np.mean(aucs)
            vocab_results.append((min_freq, max_freq, mean_auc, vocab_list, tok_count_dict))
    
    # Get best vocab bounds
    best_vocab_entry = max(vocab_results, key=lambda x: x[2])
    best_min, best_max, best_vocab_auc, best_vocab_list, best_tok_dict = best_vocab_entry
    
    if verbose:
        print(f"\nBest vocab bounds: [{best_min}, {best_max}] with AUC={best_vocab_auc:.4f}")
    
    return {
        'best_vocab_list': best_vocab_list,
        'best_tok_count_dict': best_tok_dict,
        'best_min': best_min,
        'best_max': best_max,
        'best_vocab_auc': best_vocab_auc,
        'all_vocab_results': vocab_results
    }

# Run tuning
results = find_best_vocab(
    tokenized_training_text, 
    y_train_df['Coarse Label'], 
    x_train_df,
    min_freq_list=[3,4,5, 10, 20, 50, 100],
    max_freq_list=[200, 500, 1000, 2000, 5000]
)

print(f"\n✓ Best bounds: min={results['best_min']}, max={results['best_max']}")
print(f"  Vocab size: {len(results['best_vocab_list'])}")
print(f"  Best AUC: {results['best_vocab_auc']:.4f}")


✓ Best bounds: min=3, max=1000
  Vocab size: 6587
  Best AUC: 0.7505


In [9]:
# Extract best vocab from tuning results
vocab_list = results['best_vocab_list']
tok_count_dict = results['best_tok_count_dict']

# 1) build a vocab->index mapping
vocab_dict = {w:i for i,w in enumerate(vocab_list)}

# 2) create feature matrix (dense)
V = len(vocab_list)
rows = []
for text in x_train_df['text'].values:
    vec = np.zeros(V, dtype=int)
    for tok in tokenize_text(text):
        if tok in vocab_dict:
            vec[vocab_dict[tok]] += 1
    rows.append(vec)
X_counts = np.vstack(rows)  # shape (N, V)

# 3) make a DataFrame with vocab columns and concat
vocab_cols = [f"v_{w}" for w in vocab_list]  # safe column names
vocab_df = pd.DataFrame(X_counts, columns=vocab_cols, index=x_train_df.index)
x_train_aug = pd.concat([x_train_df, vocab_df], axis=1)

print(f"Built x_train_aug with {len(vocab_cols)} vocab features")

Built x_train_aug with 6587 vocab features


In [10]:
# Load and augment test data with same vocabulary
x_test_df = pd.read_csv(os.path.join("data", "x_test.csv"))

# Create feature matrix for test data using same vocab_dict
V = len(vocab_list)
rows_test = []
for text in x_test_df['text'].values:
    vec = np.zeros(V, dtype=int)
    for tok in tokenize_text(text):
        if tok in vocab_dict:
            vec[vocab_dict[tok]] += 1
    rows_test.append(vec)
X_test_counts = np.vstack(rows_test)

# Add vocab columns to test DataFrame
vocab_test_df = pd.DataFrame(X_test_counts, columns=vocab_cols, index=x_test_df.index)
x_test_aug = pd.concat([x_test_df, vocab_test_df], axis=1)

print("Test DataFrame shape:", x_test_aug.shape)
print(x_test_aug.head())

Test DataFrame shape: (1197, 6619)
      author                                              title  passage_id  \
0  Aeschylus  The House of Atreus; Being the Agamemnon, the ...        2047   
1  Aeschylus  The House of Atreus; Being the Agamemnon, the ...        3000   
2  Aeschylus  The House of Atreus; Being the Agamemnon, the ...        9430   
3  Aeschylus  The House of Atreus; Being the Agamemnon, the ...       13796   
4  Aeschylus  The House of Atreus; Being the Agamemnon, the ...       15983   

                                                text  char_count  word_count  \
0  CHORUS    Woman, what deadly birth,  What veno...       429.0        90.0   
1  Lo, how the woman-thing, the lioness  Couched ...       477.0       106.0   
2  some dark despair of soul? CASSANDRA  Pah! the...       474.0        99.0   
3  Let my loud summons ring within the ears  Of A...       356.0        76.0   
4  HERALD  Whence thy despair, that mars the army...       465.0       107.0   

   senten

In [12]:
best_pl = make_logit_pipeline(C=1.0)
best_pl.fit(vocab_df, y_train_df['Coarse Label'])
y_test_pred_proba = best_pl.predict_proba(vocab_df)[:, 1]
with open("yproba1_test.txt", "w") as f:
    for pred in y_test_pred_proba:
        f.write(f"{pred}\n")

In [13]:
C_grid = np.logspace(-4, 4, 17)
hyper_list = []

kf = sklearn.model_selection.KFold(n_splits=10, shuffle=True, random_state=42)
for C in C_grid:
  pl = make_logit_pipeline(C=C)
  aucs = []
  for train_index, val_index in kf.split(x_train_aug):
    x_train_fold = vocab_df.iloc[train_index]
    y_train_fold = y_train_df['Coarse Label'].iloc[train_index]
    x_val_fold = vocab_df.iloc[val_index]
    y_val_fold = y_train_df['Coarse Label'].iloc[val_index]

    pl.fit(x_train_fold, y_train_fold)
    y_pred_proba = pl.predict_proba(x_val_fold)[:, 1]

    auc = sklearn.metrics.roc_auc_score(y_val_fold, y_pred_proba)
    aucs.append(auc)
  mean_auc = np.mean(aucs)
  hyper_list.append((C, pl, mean_auc))

best_C, best_pl, best_auc = max(hyper_list, key=lambda x: x[2])
print(f"Best C: {best_C}, Best AUC: {best_auc}")

Best C: 0.31622776601683794, Best AUC: 0.7541784967272052


In [18]:
best_pl.fit(vocab_df, y_train_df['Coarse Label'])
y_test_pred_proba = best_pl.predict_proba(vocab_test_df)[:, 1]
with open("yproba1_test.txt", "w") as f:
    for pred in y_test_pred_proba:
        f.write(f"{pred}\n")

In [ ]:
# roc_curve = sklearn.metrics.roc_curve(y_train_df['Coarse Label'], val_preds_prob_knn)